# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Show all record sets and their fields by `@id`

def print_record_set_overview(metadata):
    if hasattr(metadata, 'record_set'):
        for rs in metadata.record_set:
            print(f"Record Set: {getattr(rs, '@id', rs) if hasattr(rs, '@id') else rs}")
            if hasattr(rs, 'field'):
                print("  Fields:")
                for field in rs.field:
                    print(f"    - {getattr(field, '@id', field) if hasattr(field, '@id') else field}")
                print()
    else:
        print("No record sets found in metadata.")

print_record_set_overview(metadata)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# List all available record set IDs
record_set_ids = []
if hasattr(metadata, 'record_set'):
    for rs in metadata.record_set:
        if hasattr(rs, '@id'):
            record_set_ids.append(rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs.@id)
        else:
            record_set_ids.append(rs)

print("Record set IDs:")
for rid in record_set_ids:
    print("-", rid)

# Load data for all record sets (if any available)
dataframes = {}
for record_set in record_set_ids:
    print(f"Loading data for record set: {record_set}")
    try:
        records = list(dataset.records(record_set=record_set))
        dataframes[record_set] = pd.DataFrame(records)
        print(f"  Loaded {len(records)} records.")
    except Exception as e:
        print(f"  Failed to load records for {record_set}:", e)

# Pick the first record set for demonstration
if len(record_set_ids) > 0:
    main_record_set_id = record_set_ids[0]
    print(f"\nColumns in {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets found in metadata.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Example EDA using numeric and grouping fields by their `@id`
from IPython.display import display

# Set your record set ID here (chosen from above, e.g. main_record_set_id)
record_set_id = main_record_set_id if len(record_set_ids) > 0 else None
df = dataframes[record_set_id] if record_set_id else pd.DataFrame()

# You can identify numeric fields by looking at the DataFrame dtypes or by reviewing the field schema
print("DataFrame columns and types:")
display(df.dtypes)

# Automatically find a numeric column (float or int)
numeric_field = None
for col, dtype in df.dtypes.items():
    if pd.api.types.is_numeric_dtype(dtype):
        numeric_field = col
        break

if numeric_field is not None:
    print(f"Using numeric field for demo: {numeric_field}")
    threshold = df[numeric_field].mean() if not pd.isnull(df[numeric_field].mean()) else 0
    # Filter out records above mean value
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean value):")
    display(filtered_df.head())

    # Normalize the numeric field
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, norm_col]].head())

    # Try grouping by a categorical column (the first non-numeric field)
    group_field = None
    for col, dtype in df.dtypes.items():
        if not pd.api.types.is_numeric_dtype(dtype):
            group_field = col
            break
    if group_field is not None:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().sort_values(by=numeric_field, ascending=False)
        print(f"Grouped data by {group_field} (mean of {numeric_field}):")
        display(grouped_df.head())
    else:
        print("No suitable group field found (non-numeric).")
else:
    print("Could not find a numeric field for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Example: Histogram and boxplot for the chosen numeric field
import matplotlib.pyplot as plt
import seaborn as sns

if not df.empty and numeric_field:
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f'Histogram of {numeric_field}')

    plt.subplot(1,2,2)
    sns.boxplot(y=df[numeric_field].dropna())
    plt.title(f'Boxplot of {numeric_field}')
    plt.tight_layout()
    plt.show()

    # If grouping field found, plot group means
    if group_field:
        plt.figure(figsize=(10,4))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
        plt.title(f'Mean {numeric_field} by {group_field}')
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* In this notebook, we demonstrated how to use the `mlcroissant` library to:
  - Load and inspect Croissant-conformant metadata.
  - Review available record sets and field identifiers (all by `@id`).
  - Extract structured dataset records into Pandas for analysis.
  - Conduct basic EDA (filter, normalize, group) and generate simple visualizations.
* For more advanced analysis, explore additional fields, outlier removal, or modeling tasks using the rich protein mass spectrometry data accessible via the FAIR² metadata.